# Example - CITY DATA TRANSLATION DEMO

Demonstrates the core holonic library capabilities using a geographic
data scenario (following Cagel's [Vancouver example](https://ontologist.substack.com/p/the-living-graph-holons-and-the-four)).

Two holons:
  - A municipal data provider (interior uses a local schema)
  - A GeoNames-compatible consumer (boundary shapes define what it accepts)

The demo:
  1. Build both holons with TTL-defined interiors and boundaries
  2. Query the consumer's SHACL shapes to discover its expected surface
  3. Auto-generate a SPARQL CONSTRUCT portal from the discovered surface
  4. Traverse the portal to translate municipal data → GeoNames form
  5. Validate the projected output against the consumer's membrane
  6. Record provenance of the entire operation using PROV-O

In [ ]:
from holonic import (
    Holon, TransformPortal, Holarchy,
    validate_membrane, MembraneHealth,
    ProvenanceTracker,
    discover_target_shape, generate_construct_query, describe_surface,
)

## PHASE 1: Build the source holon (municipal data provider)

In [ ]:
municipal = Holon(
    iri="urn:holon:municipal:vancouver",
    label="Vancouver Municipal Data",
    depth=1,

    # Interior: what the city data system actually contains.
    # Uses a local municipal vocabulary — NOT GeoNames.
    interior_ttl="""
        @prefix muni: <urn:municipal:> .

        <urn:city:vancouver> a muni:CityRecord ;
            muni:officialName  "City of Vancouver" ;
            muni:province      "British Columbia" ;
            muni:country       "Canada" ;
            muni:censusPop     675218 ;
            muni:censusYear    2021 ;
            muni:lat           49.2827 ;
            muni:lon           -123.1207 ;
            muni:areaKm2       115.18 ;
            muni:mayor         "Ken Sim" ;
            muni:incorporated  "1886-04-06"^^xsd:date .

        <urn:city:victoria> a muni:CityRecord ;
            muni:officialName  "City of Victoria" ;
            muni:province      "British Columbia" ;
            muni:country       "Canada" ;
            muni:censusPop     91867 ;
            muni:censusYear    2021 ;
            muni:lat           48.4284 ;
            muni:lon           -123.3656 ;
            muni:areaKm2       19.47 .
    """,

    # Boundary: shapes for municipal data (what WE expect inside)
    boundary_ttl="""
        @prefix muni: <urn:municipal:> .

        <urn:shapes:muni:CityRecordShape> a sh:NodeShape ;
            sh:targetClass muni:CityRecord ;
            sh:property [
                sh:path muni:officialName ;
                sh:minCount 1 ; sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "City must have an official name."
            ] ;
            sh:property [
                sh:path muni:censusPop ;
                sh:minCount 1 ;
                sh:datatype xsd:integer ;
                sh:severity sh:Violation ;
                sh:message "Census population is required."
            ] ;
            sh:property [
                sh:path muni:lat ;
                sh:minCount 1 ;
                sh:datatype xsd:decimal ;
                sh:severity sh:Violation
            ] ;
            sh:property [
                sh:path muni:lon ;
                sh:minCount 1 ;
                sh:datatype xsd:decimal ;
                sh:severity sh:Violation
            ] .
    """,

    # Projection: external bindings (what we show outward)
    projection_ttl="""
        <urn:holon:municipal:vancouver>
            cga:bindsTo <https://sws.geonames.org/6173331/> ;
            skos:exactMatch <https://www.wikidata.org/entity/Q24639> .
    """,

    # Context: membership in the holarchy
    context_ttl="""
        <urn:holon:municipal:vancouver>
            cga:memberOf <urn:holon:geo:canada:british-columbia> .
    """,
)

# TODO replace with polars (throughout example)
print(municipal)
print(f"\n  Interior serialization (first 20 triples):")
for i, (s, p, o) in enumerate(sorted(municipal.interior)):
    if i >= 20:
        print(f"  ... and {len(municipal.interior) - 20} more")
        break
    s_short = str(s).split("/")[-1].split(":")[-1]
    p_short = str(p).split("/")[-1].split("#")[-1]
    print(f"    {s_short:30s} {p_short:20s} {o.n3()}")

## PHASE 2: Build the target holon (GeoNames-compatible consumer)

In [ ]:
geonames = Holon(
    iri="urn:holon:consumer:geonames",
    label="GeoNames Consumer",
    depth=1,

    # Interior starts empty — it will be populated by portal traversal
    interior_ttl="",

    # Boundary: THIS IS THE SELF-DESCRIBING SURFACE.
    # These shapes define what the consumer ACCEPTS.
    # A portal from any source can query these shapes to discover
    # what it needs to produce.
    boundary_ttl="""
        @prefix geo: <urn:geonames:> .

        <urn:shapes:geo:FeatureShape> a sh:NodeShape ;
            sh:targetClass geo:Feature ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "Feature must have exactly one label."
            ] ;
            sh:property [
                sh:path geo:population ;
                sh:minCount 1 ;
                sh:datatype xsd:integer ;
                sh:severity sh:Violation ;
                sh:message "Population is required."
            ] ;
            sh:property [
                sh:path geo:latitude ;
                sh:minCount 1 ;
                sh:datatype xsd:decimal ;
                sh:minInclusive -90.0 ;
                sh:maxInclusive  90.0 ;
                sh:severity sh:Violation ;
                sh:message "Latitude is required and must be in [-90, 90]."
            ] ;
            sh:property [
                sh:path geo:longitude ;
                sh:minCount 1 ;
                sh:datatype xsd:decimal ;
                sh:minInclusive -180.0 ;
                sh:maxInclusive  180.0 ;
                sh:severity sh:Violation ;
                sh:message "Longitude is required and must be in [-180, 180]."
            ] ;
            sh:property [
                sh:path geo:countryCode ;
                sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Warning ;
                sh:message "Country code is recommended."
            ] ;
            sh:property [
                sh:path geo:adminName ;
                sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Warning ;
                sh:message "Admin name (province/state) is recommended."
            ] .
    """,
)

print(geonames)

## PHASE 3: Discover the target's surface (self-describing shapes)

In [ ]:
print(describe_surface(geonames))

# Get the structured shape descriptors
shapes = discover_target_shape(geonames)
for cls, props in shapes.items():
    print(f"\n  Discovered {len(props)} properties for class {cls}:")
    for p in props:
        req = "REQUIRED" if p.is_required else "optional"
        print(f"    {p.path_local:<20s} {req:<10s} {p.severity}")

## PHASE 4: Auto-generate the portal CONSTRUCT query

In [ ]:
# Map source (municipal) properties to target (geonames) properties
# This mapping is the only manual step — it encodes domain knowledge
# about which municipal fields correspond to which GeoNames fields.
property_map = {
    "muni:officialName": "rdfs:label",
    "muni:censusPop":    "geo:population",
    "muni:lat":          "geo:latitude",
    "muni:lon":          "geo:longitude",
    "muni:country":      "geo:countryCode",
    "muni:province":     "geo:adminName",
}

construct_query = generate_construct_query(
    source_class="muni:CityRecord",
    target_class="geo:Feature",
    property_map=property_map,
    prefixes=(
        "PREFIX muni: <urn:municipal:>\n"
        "PREFIX geo:  <urn:geonames:>\n"
        "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>"
    ),
)

print("  Generated SPARQL CONSTRUCT query:\n")
for line in construct_query.strip().split("\n"):
    print(f"    {line}")

## PHASE 5: Create and traverse the TransformPortal

In [ ]:
portal = TransformPortal(
    iri="urn:portal:municipal-to-geonames",
    source=municipal,
    target=geonames,
    label="Municipal → GeoNames",
    construct_query=construct_query,
    validate_output=False,  # we'll validate manually below
)

print(f"  Created: {portal}")
print(f"  Portal registered in source boundary: {len(municipal.boundary)} triples")

# Traverse: translate municipal interior → geonames-shaped output
projected = portal.traverse(municipal.interior)

print(f"\n  Projected {len(projected)} triples:\n")
for s, p, o in sorted(projected):
    s_short = str(s).split(":")[-1]
    p_short = str(p).replace("urn:geonames:", "geo:").replace(str(
        __import__("rdflib").RDFS), "rdfs:")
    print(f"    {s_short:20s} {p_short:20s} {o.n3()}")

## PHASE 6: Validate projected output against target membrane

In [ ]:
# Inject the projected data into the target's interior
for t in projected:
    geonames.interior.add(t)

result = validate_membrane(geonames)
print(result.summary())

## PHASE 7: Demonstrate a membrane breach

In [ ]:
# Create a holon with bad data (latitude = 999)
bad_source = Holon(
    iri="urn:holon:municipal:baddata",
    label="Bad Municipal Data",
    interior_ttl="""
        @prefix muni: <urn:municipal:> .
        <urn:city:nowhere> a muni:CityRecord ;
            muni:officialName "Nowhere City" ;
            muni:censusPop 42 ;
            muni:lat 999.0 ;
            muni:lon -77.0 .
    """,
)

bad_projected = portal.traverse(bad_source.interior)

# Create a fresh target to test against
bad_target = Holon(
    iri="urn:holon:consumer:geonames-test",
    label="GeoNames Test",
    boundary_ttl=geonames.serialize_layer("boundary"),
)
for t in bad_projected:
    bad_target.interior.add(t)

bad_result = validate_membrane(bad_target)
print(bad_result.summary())

## PHASE 8: Provenance Recording

In [ ]:
prov = ProvenanceTracker(
    agent_iri="urn:agent:holonic-translator-v1",
    agent_label="Holonic Translator v1",
)

# Record the traversal
activity_id = prov.record_traversal(
    portal_iri=portal.iri,
    source=municipal,
    target=geonames,
    notes="Translated municipal city data to GeoNames feature format.",
)

# Record the validation
prov.record_validation(
    holon=geonames,
    conforms=result.conforms,
    health=result.health.value,
)

print(f"  Provenance activity: {activity_id}")
print(f"  Context graph now has {len(geonames.context)} triples")
print(f"\n  Provenance triples in target context:\n")
for s, p, o in sorted(geonames.context):
    s_short = str(s).split(":")[-1][:40]
    p_short = str(p).split("/")[-1].split("#")[-1]
    o_short = str(o)[:50]
    print(f"    {s_short:42s} {p_short:25s} {o_short}")

## PHASE 9: Holarchy Assembly

In [ ]:
holarchy = Holarchy(label="Geographic Data Holarchy")
holarchy.register(municipal)
holarchy.register(geonames)
holarchy.add_portal(portal)

print(holarchy.summary())

## Demonstrated:

- ✓ Holon construction from TTL (all four layers)
- ✓ Self-describing surface: queried target SHACL to discover
  expected properties (path, datatype, cardinality, severity)
- ✓ Auto-generated SPARQL CONSTRUCT from discovered surface +
  source→target property map
- ✓ TransformPortal traversal: translated municipal vocabulary
  to GeoNames vocabulary via CONSTRUCT
- ✓ Target membrane validation (SHACL) — both passing and breach
- ✓ PROV-O provenance recorded into target context graph
- ✓ Holarchy assembly with registered holons and portals

### Architecture:

```
Source Holon                    Target Holon
┌────────────────┐              ┌────────────────┐
│ Interior:      │  ──portal──→ │ Interior:      │
│  muni:CityRec  │  (CONSTRUCT) │  geo:Feature   │
│                │              │                │
│ Boundary:      │              │ Boundary:      │
│  CityRecShape  │   discover   │  FeatureShape  │
│                │   ←──────    │  (the surface) │
│ Projection:    │              │ Context:       │
│  GeoNames link │              │  PROV-O trail  │
└────────────────┘              └────────────────┘
```

- The target's SHACL shapes ARE the API contract.
- The CONSTRUCT query IS the translation.
- The provenance IS the audit trail.
- All live inside the hypergraph.